In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Install dependencies
!pip install ultralytics roboflow easyocr opencv-python-headless pycocotools --quiet

import os
import json
import shutil
from pathlib import Path
from ultralytics import YOLO
import cv2
import easyocr
import torch

print("✅ Setup complete!")
print(f"GPU Available: {torch.cuda.is_available()}")

In [ ]:
# === UPDATE THIS PATH ===
dataset_root = '/kaggle/input/datasets/avinash059/expirydataset'  # Change to your actual Kaggle dataset name

coco_root = Path(dataset_root)
yolo_root = Path('/kaggle/working/expiry_yolo_dataset')

# Create YOLO folder structure
for split in ['train', 'valid', 'test']:
    (yolo_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

def coco_to_yolo(coco_json_path, images_dir, output_labels_dir, output_images_dir):
    with open(coco_json_path, 'r') as f:
        data = json.load(f)
    
    # Category mapping (0: Expiry-Date, etc.)
    cat_id_to_name = {cat['id']: cat['name'] for cat in data['categories']}
    
    for ann in data['annotations']:
        image_id = ann['image_id']
        image_info = next(img for img in data['images'] if img['id'] == image_id)
        img_name = image_info['file_name']
        
        # Copy image
        src_img = images_dir / img_name
        dst_img = output_images_dir / img_name
        if src_img.exists() and not dst_img.exists():
            shutil.copy(src_img, dst_img)
        
        # Convert bbox to YOLO format (normalized)
        width, height = image_info['width'], image_info['height']
        x, y, w, h = ann['bbox']
        x_center = (x + w/2) / width
        y_center = (y + h/2) / height
        w_norm = w / width
        h_norm = h / height
        
        label_path = output_labels_dir / (Path(img_name).stem + '.txt')
        with open(label_path, 'a') as f:
            f.write(f"0 {x_center} {y_center} {w_norm} {h_norm}\n")  # class 0 = Expiry-Date

# Convert each split
for split in ['train', 'valid', 'test']:
    coco_json = coco_root / split / '_annotations.coco.json'
    images_dir = coco_root / split
    if coco_json.exists():
        print(f"Converting {split} split...")
        coco_to_yolo(
            coco_json,
            images_dir,
            yolo_root / 'labels' / split,
            yolo_root / 'images' / split
        )

print("✅ Conversion completed!")
print(f"Dataset ready at: {yolo_root}")

In [ ]:
data_yaml_content = f"""
train: {yolo_root}/images/train
val: {yolo_root}/images/valid
test: {yolo_root}/images/test

nc: 1
names: ['Expiry-Date']
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("✅ data.yaml created")

In [ ]:
import cv2
import easyocr
from PIL import Image, ImageEnhance, ImageFilter

reader = easyocr.Reader(['en'], gpu=True)

def enhanced_expiry_ocr(image_path, det_model):
    results = det_model(image_path, conf=0.4, iou=0.45)
    
    if len(results[0].boxes) == 0:
        return {"status": "No date detected", "text": ""}
    
    # Get the best detection (highest confidence)
    best_box = results[0].boxes[results[0].boxes.conf.argmax()]
    box = best_box.xyxy[0].cpu().numpy().astype(int)
    x1, y1, x2, y2 = box
    
    img = cv2.imread(image_path)
    cropped = img[y1:y2, x1:x2]
    
    # Multiple preprocessing techniques
    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    
    # Enhancement 1: Contrast + Brightness
    enhanced = cv2.convertScaleAbs(gray, alpha=1.8, beta=30)
    
    # Enhancement 2: Denoising
    denoised = cv2.fastNlMeansDenoising(enhanced)
    
    # Enhancement 3: Sharpening
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(denoised, -1, kernel)
    
    # Try OCR on multiple versions
    texts = []
    for img_version in [denoised, sharpened, enhanced]:
        text = ' '.join(reader.readtext(img_version, detail=0, paragraph=True))
        if text:
            texts.append(text)
    
    final_text = max(texts, key=len) if texts else ""
    
    return {
        "status": "Success",
        "text": final_text.strip(),
        "confidence": float(best_box.conf)
    }




In [ ]:
# === IMPROVED TRAINING BLOCK (Replace your old train cell) ===
from ultralytics import YOLO

# Use a stronger base model
model = YOLO('yolov8m.pt')        # Better than yolov8s  (or try 'yolov11m.pt')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=80,                    # Increased
    imgsz=640,                    # Try 800 if you have GPU time
    batch=16,                     # Reduce to 8 if OOM error
    name='expiry_detector_improved',
    patience=25,
    pretrained=True,
    
    # Optimizer & Learning Rate
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    
    # Strong Augmentation (Very Important for expiry dates)
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.6,
    shear=3.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    
    # Other good settings
    conf=0.25,
    iou=0.45,
    box=7.5,          # Box loss gain
    cls=0.5           # Class loss gain
)

print("✅ Improved Training Completed!")
print("Best model saved at:", '/kaggle/working/runs/detect/expiry_detector_improved/weights/best.pt')

In [ ]:
import shutil
from pathlib import Path

# Find the best model
weights_path = Path('/kaggle/working/runs/detect/expiry_detector_improved-3/weights/best.pt')

if weights_path.exists():
    print("✅ Found best.pt at:", weights_path)
    
    # Create a zip for easy download
    shutil.make_archive('/kaggle/working/expiry_model', 'zip', 
                       root_dir=weights_path.parent, base_dir='best.pt')
    
    print("✅ Model zipped at: /kaggle/working/expiry_model.zip")
else:
    print("❌ Model not found. Check the training path below:")
    # List all runs
    !ls -R /kaggle/working/runs/

In [ ]:
model = YOLO('/kaggle/working/runs/detect/expiry_detector_improved-3/weights/best.pt')

# Test on a few images
test_images = list(Path('/kaggle/input/datasets/avinash059/expirydataset/train').glob('*.jpg'))[:20]

for img_path in test_images:
    results = model(img_path)
    results[0].show()  # Displays with bounding box
    print("Detected expiry regions in:", img_path.name)

In [ ]:
# === UPDATED FOR YOUR KAGGLE DATASET ===
dataset_root = '/kaggle/input/datasets/avinash059/expirydataset'   # ← CHANGE THIS to exact name shown in right sidebar

# Check actual structure
print("Dataset contents:")
!ls -l {dataset_root}

coco_root = Path(dataset_root)
yolo_root = Path('/kaggle/working/expiry_yolo_dataset')

# Create YOLO structure
for split in ['train', 'valid', 'test']:
    (yolo_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

def coco_to_yolo(coco_json_path, images_dir, output_labels_dir, output_images_dir):
    with open(coco_json_path, 'r') as f:
        data = json.load(f)
    
    cat_id_to_name = {cat['id']: cat['name'] for cat in data.get('categories', [])}
    print("Categories found:", cat_id_to_name)
    
    for ann in data.get('annotations', []):
        image_id = ann['image_id']
        image_info = next((img for img in data.get('images', []) if img['id'] == image_id), None)
        if not image_info:
            continue
            
        img_name = image_info['file_name']
        
        # Copy image
        src_img = images_dir / img_name
        dst_img = output_images_dir / img_name
        if src_img.exists() and not dst_img.exists():
            shutil.copy(src_img, dst_img)
        
        # Convert bbox to YOLO format
        width, height = image_info['width'], image_info['height']
        x, y, w, h = ann['bbox']
        x_center = (x + w/2) / width
        y_center = (y + h/2) / height
        w_norm = w / width
        h_norm = h / height
        
        label_path = output_labels_dir / (Path(img_name).stem + '.txt')
        with open(label_path, 'a') as f:
            f.write(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

# Convert all splits
for split in ['train', 'valid', 'test']:
    coco_json = coco_root / split / '_annotations.coco.json'
    images_dir = coco_root / split
    if coco_json.exists():
        print(f"✅ Converting {split} split...")
        coco_to_yolo(coco_json, images_dir, yolo_root / 'labels' / split, yolo_root / 'images' / split)
    else:
        print(f"⚠️ {split} folder not found")

print("✅ Conversion Done! Dataset at:", yolo_root)